In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 09 — FINAL VIDEO INFERENCE
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms

from PIL import Image

PROJECT_DIR = "/content/drive/MyDrive/Multimedia_Authentication"

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models"
)

GATED_CHECKPOINT = os.path.join(
    MODEL_DIR,
    "gated_fusion_resnet18_auc_best.pth"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("FINAL VIDEO INFERENCE")
print("=" * 60)

print("Project:", PROJECT_DIR)
print("Checkpoint:", GATED_CHECKPOINT)
print("Device:", DEVICE)

print("\nCheckpoint exists:",
      os.path.exists(GATED_CHECKPOINT))

FINAL VIDEO INFERENCE
Project: /content/drive/MyDrive/Multimedia_Authentication
Checkpoint: /content/drive/MyDrive/Multimedia_Authentication/models/gated_fusion_resnet18_auc_best.pth
Device: cpu

Checkpoint exists: True


In [6]:

print("Available model files:")
print("=" * 60)

for root, dirs, files in os.walk(MODEL_DIR):

    for file in files:

        path = os.path.join(root, file)

        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(
            f"{file:<45} "
            f"{size_mb:>8.2f} MB"
        )

Available model files:
fusion_resnet18_auc_best.pth                     86.44 MB
gated_fusion_resnet18_auc_best.pth                1.01 MB


In [7]:
print("Searching for all .pth checkpoints...")
print("=" * 60)

for root, dirs, files in os.walk(PROJECT_DIR):

    for file in files:

        if file.endswith(".pth"):

            path = os.path.join(root, file)

            size_mb = os.path.getsize(path) / (1024 * 1024)

            print(f"{file:<50} {size_mb:>8.2f} MB")
            print("  ", path)

Searching for all .pth checkpoints...
spatial_resnet18_best.pth                            128.05 MB
   /content/drive/MyDrive/Multimedia_Authentication/spatial_resnet18_best.pth
frequency_resnet18_best.pth                          128.05 MB
   /content/drive/MyDrive/Multimedia_Authentication/frequency_resnet18_best.pth
fusion_resnet18_best_f1.pth                           86.44 MB
   /content/drive/MyDrive/Multimedia_Authentication/fusion_resnet18_best_f1.pth
fusion_resnet18_auc_best.pth                          86.44 MB
   /content/drive/MyDrive/Multimedia_Authentication/models/fusion_resnet18_auc_best.pth
gated_fusion_resnet18_auc_best.pth                     1.01 MB
   /content/drive/MyDrive/Multimedia_Authentication/models/gated_fusion_resnet18_auc_best.pth


In [12]:
# INSPECT BRANCH CHECKPOINT STRUCTURE

def inspect_checkpoint(path, name):

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint type:", type(checkpoint))

    if isinstance(checkpoint, dict):

        print("\nTop-level keys:")
        for key in checkpoint.keys():
            print(" ", key)

        if "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        else:
            state_dict = checkpoint

        print("\nFirst 20 state-dict keys:")
        for key in list(state_dict.keys())[:20]:
            print(" ", key)

        print("\nTotal parameters:", len(state_dict))


inspect_checkpoint(
    SPATIAL_CHECKPOINT,
    "SPATIAL MODEL"
)

inspect_checkpoint(
    FREQUENCY_CHECKPOINT,
    "FREQUENCY MODEL"
)


SPATIAL MODEL
Checkpoint type: <class 'dict'>

Top-level keys:
  epoch
  model_state_dict
  optimizer_state_dict
  val_f1

First 20 state-dict keys:
  backbone.0.weight
  backbone.1.weight
  backbone.1.bias
  backbone.1.running_mean
  backbone.1.running_var
  backbone.1.num_batches_tracked
  backbone.4.0.conv1.weight
  backbone.4.0.bn1.weight
  backbone.4.0.bn1.bias
  backbone.4.0.bn1.running_mean
  backbone.4.0.bn1.running_var
  backbone.4.0.bn1.num_batches_tracked
  backbone.4.0.conv2.weight
  backbone.4.0.bn2.weight
  backbone.4.0.bn2.bias
  backbone.4.0.bn2.running_mean
  backbone.4.0.bn2.running_var
  backbone.4.0.bn2.num_batches_tracked
  backbone.4.1.conv1.weight
  backbone.4.1.bn1.weight

Total parameters: 122

FREQUENCY MODEL
Checkpoint type: <class 'dict'>

Top-level keys:
  epoch
  model_state_dict
  optimizer_state_dict
  val_f1

First 20 state-dict keys:
  backbone.0.weight
  backbone.1.weight
  backbone.1.bias
  backbone.1.running_mean
  backbone.1.running_var
  backbone

In [15]:
#  LOAD TRAINED SPATIAL + FREQUENCY MODELS

import torchvision.models as models


class ResNet18Binary(nn.Module):

    def __init__(self):
        super().__init__()

        resnet = models.resnet18(weights=None)

        # Remove the original classification layer
        self.backbone = nn.Sequential(
            *list(resnet.children())[:-1]
        )

        self.classifier = nn.Linear(512, 2)

    def forward(self, x):
        features = self.backbone(x)
        features = torch.flatten(features, 1)
        return self.classifier(features)


def load_branch_model(checkpoint_path):

    model = ResNet18Binary().to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    return model


spatial_model = load_branch_model(
    SPATIAL_CHECKPOINT
)

frequency_model = load_branch_model(
    FREQUENCY_CHECKPOINT
)

print("Spatial ResNet18 loaded successfully.")
print("Frequency ResNet18 loaded successfully.")

Spatial ResNet18 loaded successfully.
Frequency ResNet18 loaded successfully.


In [16]:
# 512-D FEATURE EXTRACTORS

spatial_features = spatial_model.backbone
frequency_features = frequency_model.backbone

spatial_features.eval()
frequency_features.eval()

print("Spatial feature extractor: 512-D")
print("Frequency feature extractor: 512-D")

Spatial feature extractor: 512-D
Frequency feature extractor: 512-D


In [17]:
#  FEATURE EXTRACTION SANITY CHECK


dummy = torch.randn(
    1, 3, 224, 224
).to(DEVICE)

with torch.no_grad():

    spatial_test = spatial_features(dummy)
    frequency_test = frequency_features(dummy)

    spatial_test = torch.flatten(
        spatial_test, 1
    )

    frequency_test = torch.flatten(
        frequency_test, 1
    )

print("Spatial feature shape:",
      spatial_test.shape)

print("Frequency feature shape:",
      frequency_test.shape)

Spatial feature shape: torch.Size([1, 512])
Frequency feature shape: torch.Size([1, 512])


In [18]:
# VIDEO FRAME + FACE EXTRACTION

from google.colab import files

FACE_CASCADE = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"

face_detector = cv2.CascadeClassifier(FACE_CASCADE)


def extract_video_faces(video_path, num_frames=8):

    cap = cv2.VideoCapture(video_path)

    total_frames = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if total_frames <= 0:
        cap.release()
        raise ValueError("Could not read video.")

    frame_indices = np.linspace(
        0,
        total_frames - 1,
        num_frames,
        dtype=int
    )

    faces = []

    for idx in frame_indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(idx)
        )

        ret, frame = cap.read()

        if not ret:
            continue

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        detected = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        if len(detected) == 0:
            continue

        # Select largest detected face
        x, y, w, h = max(
            detected,
            key=lambda box: box[2] * box[3]
        )

        face = frame[
            y:y+h,
            x:x+w
        ]

        face = cv2.cvtColor(
            face,
            cv2.COLOR_BGR2RGB
        )

        faces.append(
            Image.fromarray(face)
        )

    cap.release()

    return faces

In [35]:
uploaded = files.upload()

video_path = next(iter(uploaded.keys()))

print("Video:", video_path)

faces = extract_video_faces(
    video_path,
    num_frames=8
)

print("Faces extracted:", len(faces))

Saving Screen Recording 2026-10-03 185055.mp4 to Screen Recording 2026-10-03 185055.mp4
Video: Screen Recording 2026-10-03 185055.mp4
Faces extracted: 8


In [36]:
import pywt

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


def dwt_transform(image_tensor):

    # Convert normalized tensor back to image-like range
    image_np = image_tensor.clone()

    mean = torch.tensor(
        [0.485, 0.456, 0.406]
    ).view(3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225]
    ).view(3, 1, 1)

    image_np = image_np * std + mean

    image_np = image_np.clamp(0, 1)

    image_np = image_np.permute(
        1, 2, 0
    ).cpu().numpy()

    channels = []

    for c in range(3):

        LL, (LH, HL, HH) = pywt.dwt2(
            image_np[:, :, c],
            "haar"
        )

        high_freq = (
            np.abs(LH)
            + np.abs(HL)
            + np.abs(HH)
        ) / 3.0

        channels.append(high_freq)

    freq = np.stack(
        channels,
        axis=2
    )

    # Normalize frequency representation
    freq_min = freq.min()
    freq_max = freq.max()

    if freq_max > freq_min:

        freq = (
            freq - freq_min
        ) / (
            freq_max - freq_min
        )

    freq = cv2.resize(
        freq,
        (224, 224)
    )

    freq_tensor = torch.tensor(
        freq,
        dtype=torch.float32
    ).permute(2, 0, 1)

    freq_tensor = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )(freq_tensor)

    return freq_tensor


print("Preprocessing ready.")
print("DWT transform ready.")

Preprocessing ready.
DWT transform ready.


In [37]:
# EXTRACT VIDEO FEATUREs

def extract_features_from_faces(faces):

    spatial_batch = []
    frequency_batch = []

    for face in faces:

        # Spatial representation
        spatial_img = transform(face)

        # Frequency representation
        frequency_img = dwt_transform(
            spatial_img
        )

        spatial_batch.append(
            spatial_img
        )

        frequency_batch.append(
            frequency_img
        )

    spatial_batch = torch.stack(
        spatial_batch
    ).to(DEVICE)

    frequency_batch = torch.stack(
        frequency_batch
    ).to(DEVICE)

    with torch.no_grad():

        spatial_feat = spatial_features(
            spatial_batch
        )

        frequency_feat = frequency_features(
            frequency_batch
        )

        spatial_feat = torch.flatten(
            spatial_feat,
            1
        )

        frequency_feat = torch.flatten(
            frequency_feat,
            1
        )

    # Temporal aggregation
    spatial_feat = spatial_feat.mean(
        dim=0,
        keepdim=True
    )

    frequency_feat = frequency_feat.mean(
        dim=0,
        keepdim=True
    )

    return spatial_feat, frequency_feat


spatial_video_feat, frequency_video_feat = \
    extract_features_from_faces(faces)

print(
    "Spatial feature:",
    spatial_video_feat.shape
)

print(
    "Frequency feature:",
    frequency_video_feat.shape
)

Spatial feature: torch.Size([1, 512])
Frequency feature: torch.Size([1, 512])


In [38]:
# CELL 12 — FINAL VIDEO PREDICTION
with torch.no_grad():

    logits, alpha = model(
        spatial_video_feat,
        frequency_video_feat,
        return_gate=True
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )[0]

    prediction = torch.argmax(
        probabilities
    ).item()

    confidence = probabilities[
        prediction
    ].item()

    gate_value = alpha.item()


label = "REAL" if prediction == 0 else "FAKE"

print("=" * 60)
print("FINAL VIDEO AUTHENTICATION RESULT")
print("=" * 60)

print(f"Prediction : {label}")
print(f"Confidence : {confidence * 100:.2f}%")
print(f"Gate α     : {gate_value:.4f}")

print("=" * 60)

FINAL VIDEO AUTHENTICATION RESULT
Prediction : FAKE
Confidence : 91.07%
Gate α     : 1.0000


In [29]:
# INSPECT SAVED FACE SEQUENCES

real_file = os.path.join(
    FACES_DIR,
    "original__916.npz"
)

fake_file = os.path.join(
    FACES_DIR,
    "FaceSwap__280_249.npz"
)


def inspect_face_file(path):

    data = np.load(path)

    print("\nFile:", os.path.basename(path))
    print("Keys:", data.files)

    for key in data.files:
        arr = data[key]
        print(
            f"{key}: shape={arr.shape}, dtype={arr.dtype}"
        )


inspect_face_file(real_file)
inspect_face_file(fake_file)


File: original__916.npz
Keys: ['frames']
frames: shape=(8, 224, 224, 3), dtype=uint8

File: FaceSwap__280_249.npz
Keys: ['frames']
frames: shape=(8, 224, 224, 3), dtype=uint8


In [42]:
def predict_face_sequence(path):

    data = np.load(path)

    frames = data["frames"]

    spatial_batch = []
    frequency_batch = []

    for frame in frames:

        # uint8 RGB → PIL
        image = Image.fromarray(frame)

        # Spatial input
        spatial_img = transform(image)

        # Frequency input
        frequency_img = dwt_transform(
            spatial_img
        )

        spatial_batch.append(spatial_img)
        frequency_batch.append(frequency_img)

    spatial_batch = torch.stack(
        spatial_batch
    ).to(DEVICE)

    frequency_batch = torch.stack(
        frequency_batch
    ).to(DEVICE)

    with torch.no_grad():

        # 8-frame spatial features
        spatial_feat = spatial_features(
            spatial_batch
        )

        # 8-frame frequency features
        frequency_feat = frequency_features(
            frequency_batch
        )

        spatial_feat = torch.flatten(
            spatial_feat,
            1
        )

        frequency_feat = torch.flatten(
            frequency_feat,
            1
        )

        # Temporal mean — same as training
        spatial_feat = spatial_feat.mean(
            dim=0,
            keepdim=True
        )

        frequency_feat = frequency_feat.mean(
            dim=0,
            keepdim=True
        )

        # Adaptive fusion
        logits, alpha = model(
            spatial_feat,
            frequency_feat,
            return_gate=True
        )

        probs = torch.softmax(
            logits,
            dim=1
        )[0]

        prediction = torch.argmax(
            probs
        ).item()

        confidence = probs[
            prediction
        ].item()

    label = "REAL" if prediction == 0 else "FAKE"

    return label, confidence, alpha.item()


real_result = predict_face_sequence(
    real_file
)

fake_result = predict_face_sequence(
    fake_file
)



print("REAL SAMPLE")
print("File       :", os.path.basename(real_file))
print("Prediction :", real_result[0])
print("Confidence :", f"{real_result[1] * 100:.2f}%")
print("Gate α     :", f"{real_result[2]:.4f}")

print()

print("FAKE SAMPLE")
print("File       :", os.path.basename(fake_file))
print("Prediction :", fake_result[0])
print("Confidence :", f"{fake_result[1] * 100:.2f}%")
print("Gate α     :", f"{fake_result[2]:.4f}")

REAL SAMPLE
File       : original__916.npz
Prediction : REAL
Confidence : 98.58%
Gate α     : 1.0000

FAKE SAMPLE
File       : FaceSwap__280_249.npz
Prediction : FAKE
Confidence : 97.19%
Gate α     : 1.0000


In [43]:
# FINAL REUSABLE VIDEO PREDICTOR

def predict_video(video_path):

    faces = extract_video_faces(
        video_path,
        num_frames=8
    )

    if len(faces) == 0:
        return {
            "status": "error",
            "message": "No face detected in the video."
        }

    spatial_feat, frequency_feat = \
        extract_features_from_faces(faces)

    with torch.no_grad():

        logits, alpha = model(
            spatial_feat,
            frequency_feat,
            return_gate=True
        )

        probs = torch.softmax(
            logits,
            dim=1
        )[0]

        prediction = torch.argmax(
            probs
        ).item()

        confidence = probs[
            prediction
        ].item()

        gate_value = alpha.item()

    label = (
        "REAL"
        if prediction == 0
        else "FAKE"
    )

    return {
        "status": "success",
        "prediction": label,
        "confidence": confidence,
        "gate_alpha": gate_value,
        "faces_detected": len(faces)
    }


print("Reusable video predictor ready.")

Reusable video predictor ready.


In [ ]:
result = predict_video(
    external_video
)

print("=" * 60)
print("FINAL INFERENCE")
print("=" * 60)

for key, value in result.items():
    print(f"{key:<18}: {value}")

print("=" * 60)